# Control λ_U = 0 — ¿de dónde viene la ganancia de SSL en UNM?

## Qué prueba este notebook

Las corridas SSL y la supervisada se diferencian en **dos** cosas, no en una:

1. La pérdida no supervisada (lo que el paper quiere medir).
2. Que las imágenes no etiquetadas **pasan por la red desde la época 0**.

El punto 2 no es intencional. En `src/train.py` la variable `use_semi` no tiene compuerta de
época: el lote no etiquetado se toma, el estudiante hace forward sobre él y la EMA del profesor
se actualiza desde la primera época. La rampa (`_ramp_weight`) escala **solo el peso de la
pérdida**. Con EfficientNet-B3, que usa BatchNorm, y el modelo en modo `train`, eso significa que
las estadísticas de normalización se adaptan a los datos no etiquetados aunque λ_U = 0.

Este control apaga la pérdida y deja todo lo demás igual.

## Por qué no hace falta tocar el código

Con `lambda_u = 0.0` y `use_semi = True`:

| | Sigue ocurriendo |
|---|---|
| Se toma el lote no etiquetado | sí |
| Forward del estudiante sobre la vista fuerte → BatchNorm se adapta | sí |
| Actualización EMA del profesor | sí |
| `loss + lambda_u_t * unsup_loss` aporta gradiente | **no** (`lambda_u_t ≡ 0`) |

`_ramp_weight(..., max_weight=0.0)` devuelve 0 en toda época y no hay ninguna guarda que cambie
de rama. El camino de código es idéntico, así que el orden de los datos y las aumentaciones
coinciden con los de la corrida real: es un contrafactual emparejado, no una corrida parecida.

**No se modifica `train.py` ni ningún notebook existente.** La salida va a
`runs_control_lambda0/`, un directorio nuevo.

## Cómo leer el resultado

Referencias de `runs_final_v1/` (media de 3 semillas, F1 de test):

| condición | F1 |
|---|---|
| supervisado (sin datos no etiquetados) | **0.801** |
| MT r=10 | 0.850 |
| MT all-lateral | **0.860** |

| Si el control da... | Significa | Qué hacer con el paper |
|---|---|---|
| ≈ 0.801 | La ganancia viene de la pérdida SSL | La afirmación causal se sostiene. Reportar el control como evidencia a favor. |
| ≈ 0.850–0.860 | La ganancia viene de pasar datos no etiquetados por la red | Reescribir la atribución causal. Sigue siendo un resultado publicable e interesante, pero es otro mecanismo. |
| intermedio | Contribuyen las dos cosas | Reportar la descomposición: cuánto aporta cada una. |

Los tres desenlaces son publicables. El que no es publicable es no saberlo.


In [ ]:
# ============================================================
# SETUP — correr una vez tras cada reinicio del runtime
# No entrena nada. Elegir abajo una celda de corrida.
# ============================================================
from google.colab import drive
drive.mount("/content/drive")

!git clone https://github.com/sebastianquispearias/tesis-seg.git
%cd tesis-seg
!pip install -q -r requirements.txt

import torch, sys
print("Python    :", sys.version)
print("torch     :", torch.__version__)
print("CUDA      :", torch.version.cuda)
!git log --oneline -1
!pip show albumentations | grep Version
!nvidia-smi | grep -E "NVIDIA|Driver Version|CUDA Version"

import sys
sys.path.append("/content/tesis-seg")

import json, os, time, glob

from src.defaults import get_default_config, summarize_config
from src.augmentations import (
    get_supervised_train_augmentation,
    get_weak_augmentation,
    get_strong_augmentation,
)
from src.datasets import (
    build_supervised_datasets,
    build_unlabeled_datasets,
    build_dataloaders,
)
from src.train import run_training
from src.evaluate import evaluate_checkpoint

# --- Comprobacion de que lambda_u=0 hace lo que creemos ---
from src.train import _ramp_weight
assert _ramp_weight(epoch=0,   start_epoch=15, warmup_epochs=20, max_weight=0.0) == 0.0
assert _ramp_weight(epoch=100, start_epoch=15, warmup_epochs=20, max_weight=0.0) == 0.0
assert _ramp_weight(epoch=100, start_epoch=15, warmup_epochs=20, max_weight=0.05) == 0.05
print("OK: con max_weight=0.0 la rampa devuelve 0 en toda epoca; el codigo no se toco.")
# --- GUARDIAN: el clon debe traer el congelado de BatchNorm ---
# El 2026-08-16 se perdieron 20 runs porque Colab clono una version del repo
# anterior al commit 8fe374d: freeze_bn_on_unlabeled se ignoro en silencio y
# los resultados salieron byte a byte identicos a los del control sin congelar.
from src import train as _t
assert hasattr(_t, "_bn_congelado"), (
    "CODIGO VIEJO: el clon no tiene _bn_congelado. Borra /content/tesis-seg, "
    "vuelve a clonar y reinicia el entorno de ejecucion."
)
print("OK: el codigo clonado incluye el congelado de BatchNorm.")


---
## Corridas

Cada celda es independiente y tiene skip-logic: se puede reejecutar sin repetir trabajo.


### control_lambda0_r10

Control de `runs_final_v1/mean_teacher_r10/`. Pool: `unlabeling_r10_max0/images`


In [ ]:
# === CONTROL 1/6: control_lambda0_r10/seed_0 ===
# Identico a runs_final_v1/mean_teacher_r10/seed_0 SALVO lambda_u = 0.0 y exp_dir.
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "control_lambda0_r10"
_SEED     = 0
# Directorio NUEVO: no toca runs_final_v1/ ni ningun otro resultado existente.
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_control_lambda0/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised — UNICA DIFERENCIA con la corrida final: lambda_u = 0.0
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True     # se mantiene: el forward no etiquetado y la EMA deben ocurrir
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.0      # <<<<<< el control
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeling_r10_max0/images"

cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- skip-logic (mismo criterio que 07_unm_final_rerun) ---
_exp_dir = cfg["exp_dir"]
_best_path_skip    = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
_has_best    = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report  = len(glob.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all  = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado")

if not _skip_all:
    assert cfg["lambda_u"] == 0.0, "el control exige lambda_u = 0"
    assert cfg["use_semi"] is True, "use_semi debe seguir en True"
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )

    loaders = build_dataloaders(
        cfg, train_ds=train_ds, val_ds=val_ds, test_ds=test_ds,
        unlabeled_ds=unlabeled_ds, temporal_unlab_ds=temporal_unlab_ds,
    )

    if _eval_only:
        from src.models import create_model
        _m = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
        results = evaluate_checkpoint(cfg, _m, loaders, _best_path_skip, [])
    else:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg, artifacts["model"], loaders,
            artifacts["best_path"], artifacts["history"],
        )
    print(results)


In [ ]:
# === CONTROL 2/6: control_lambda0_r10/seed_1 ===
# Identico a runs_final_v1/mean_teacher_r10/seed_1 SALVO lambda_u = 0.0 y exp_dir.
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "control_lambda0_r10"
_SEED     = 1
# Directorio NUEVO: no toca runs_final_v1/ ni ningun otro resultado existente.
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_control_lambda0/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised — UNICA DIFERENCIA con la corrida final: lambda_u = 0.0
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True     # se mantiene: el forward no etiquetado y la EMA deben ocurrir
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.0      # <<<<<< el control
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeling_r10_max0/images"

cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- skip-logic (mismo criterio que 07_unm_final_rerun) ---
_exp_dir = cfg["exp_dir"]
_best_path_skip    = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
_has_best    = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report  = len(glob.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all  = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado")

if not _skip_all:
    assert cfg["lambda_u"] == 0.0, "el control exige lambda_u = 0"
    assert cfg["use_semi"] is True, "use_semi debe seguir en True"
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )

    loaders = build_dataloaders(
        cfg, train_ds=train_ds, val_ds=val_ds, test_ds=test_ds,
        unlabeled_ds=unlabeled_ds, temporal_unlab_ds=temporal_unlab_ds,
    )

    if _eval_only:
        from src.models import create_model
        _m = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
        results = evaluate_checkpoint(cfg, _m, loaders, _best_path_skip, [])
    else:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg, artifacts["model"], loaders,
            artifacts["best_path"], artifacts["history"],
        )
    print(results)


In [ ]:
# === CONTROL 3/6: control_lambda0_r10/seed_2 ===
# Identico a runs_final_v1/mean_teacher_r10/seed_2 SALVO lambda_u = 0.0 y exp_dir.
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "control_lambda0_r10"
_SEED     = 2
# Directorio NUEVO: no toca runs_final_v1/ ni ningun otro resultado existente.
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_control_lambda0/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised — UNICA DIFERENCIA con la corrida final: lambda_u = 0.0
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True     # se mantiene: el forward no etiquetado y la EMA deben ocurrir
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.0      # <<<<<< el control
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeling_r10_max0/images"

cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- skip-logic (mismo criterio que 07_unm_final_rerun) ---
_exp_dir = cfg["exp_dir"]
_best_path_skip    = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
_has_best    = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report  = len(glob.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all  = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado")

if not _skip_all:
    assert cfg["lambda_u"] == 0.0, "el control exige lambda_u = 0"
    assert cfg["use_semi"] is True, "use_semi debe seguir en True"
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )

    loaders = build_dataloaders(
        cfg, train_ds=train_ds, val_ds=val_ds, test_ds=test_ds,
        unlabeled_ds=unlabeled_ds, temporal_unlab_ds=temporal_unlab_ds,
    )

    if _eval_only:
        from src.models import create_model
        _m = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
        results = evaluate_checkpoint(cfg, _m, loaders, _best_path_skip, [])
    else:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg, artifacts["model"], loaders,
            artifacts["best_path"], artifacts["history"],
        )
    print(results)


### control_lambda0_all_lateral

Control de `runs_final_v1/mean_teacher_all_lateral/`. Pool: `unlabeling_all_lateral/images`


In [ ]:
# === CONTROL 4/6: control_lambda0_all_lateral/seed_0 ===
# Identico a runs_final_v1/mean_teacher_all_lateral/seed_0 SALVO lambda_u = 0.0 y exp_dir.
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "control_lambda0_all_lateral"
_SEED     = 0
# Directorio NUEVO: no toca runs_final_v1/ ni ningun otro resultado existente.
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_control_lambda0/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised — UNICA DIFERENCIA con la corrida final: lambda_u = 0.0
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True     # se mantiene: el forward no etiquetado y la EMA deben ocurrir
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.0      # <<<<<< el control
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeling_all_lateral/images"

cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- skip-logic (mismo criterio que 07_unm_final_rerun) ---
_exp_dir = cfg["exp_dir"]
_best_path_skip    = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
_has_best    = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report  = len(glob.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all  = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado")

if not _skip_all:
    assert cfg["lambda_u"] == 0.0, "el control exige lambda_u = 0"
    assert cfg["use_semi"] is True, "use_semi debe seguir en True"
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )

    loaders = build_dataloaders(
        cfg, train_ds=train_ds, val_ds=val_ds, test_ds=test_ds,
        unlabeled_ds=unlabeled_ds, temporal_unlab_ds=temporal_unlab_ds,
    )

    if _eval_only:
        from src.models import create_model
        _m = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
        results = evaluate_checkpoint(cfg, _m, loaders, _best_path_skip, [])
    else:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg, artifacts["model"], loaders,
            artifacts["best_path"], artifacts["history"],
        )
    print(results)


In [ ]:
# === CONTROL 5/6: control_lambda0_all_lateral/seed_1 ===
# Identico a runs_final_v1/mean_teacher_all_lateral/seed_1 SALVO lambda_u = 0.0 y exp_dir.
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "control_lambda0_all_lateral"
_SEED     = 1
# Directorio NUEVO: no toca runs_final_v1/ ni ningun otro resultado existente.
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_control_lambda0/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised — UNICA DIFERENCIA con la corrida final: lambda_u = 0.0
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True     # se mantiene: el forward no etiquetado y la EMA deben ocurrir
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.0      # <<<<<< el control
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeling_all_lateral/images"

cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- skip-logic (mismo criterio que 07_unm_final_rerun) ---
_exp_dir = cfg["exp_dir"]
_best_path_skip    = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
_has_best    = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report  = len(glob.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all  = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado")

if not _skip_all:
    assert cfg["lambda_u"] == 0.0, "el control exige lambda_u = 0"
    assert cfg["use_semi"] is True, "use_semi debe seguir en True"
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )

    loaders = build_dataloaders(
        cfg, train_ds=train_ds, val_ds=val_ds, test_ds=test_ds,
        unlabeled_ds=unlabeled_ds, temporal_unlab_ds=temporal_unlab_ds,
    )

    if _eval_only:
        from src.models import create_model
        _m = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
        results = evaluate_checkpoint(cfg, _m, loaders, _best_path_skip, [])
    else:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg, artifacts["model"], loaders,
            artifacts["best_path"], artifacts["history"],
        )
    print(results)


In [ ]:
# === CONTROL 6/6: control_lambda0_all_lateral/seed_2 ===
# Identico a runs_final_v1/mean_teacher_all_lateral/seed_2 SALVO lambda_u = 0.0 y exp_dir.
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "control_lambda0_all_lateral"
_SEED     = 2
# Directorio NUEVO: no toca runs_final_v1/ ni ningun otro resultado existente.
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_control_lambda0/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised — UNICA DIFERENCIA con la corrida final: lambda_u = 0.0
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True     # se mantiene: el forward no etiquetado y la EMA deben ocurrir
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.0      # <<<<<< el control
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeling_all_lateral/images"

cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- skip-logic (mismo criterio que 07_unm_final_rerun) ---
_exp_dir = cfg["exp_dir"]
_best_path_skip    = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
_has_best    = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report  = len(glob.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all  = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado")

if not _skip_all:
    assert cfg["lambda_u"] == 0.0, "el control exige lambda_u = 0"
    assert cfg["use_semi"] is True, "use_semi debe seguir en True"
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )

    loaders = build_dataloaders(
        cfg, train_ds=train_ds, val_ds=val_ds, test_ds=test_ds,
        unlabeled_ds=unlabeled_ds, temporal_unlab_ds=temporal_unlab_ds,
    )

    if _eval_only:
        from src.models import create_model
        _m = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
        results = evaluate_checkpoint(cfg, _m, loaders, _best_path_skip, [])
    else:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg, artifacts["model"], loaders,
            artifacts["best_path"], artifacts["history"],
        )
    print(results)


---
## Resumen y lectura del resultado


In [ ]:
# === RESUMEN (solo lectura, no entrena nada) ===
import json, glob, os, statistics as st

ROOT = "/content/drive/MyDrive/UNM_vertebras_seg_v3"

def f1_of(root, cond):
    vals = []
    for p in sorted(glob.glob(os.path.join(ROOT, root, cond, "seed_*", "*_run_report.json"))):
        d = json.load(open(p))
        v = d.get("test_metrics", {}).get("sample_mean_f1")
        if v is not None:
            vals.append(v)
    return vals

filas = [
    ("supervisado (sin datos no etiquetados)", "runs_final_v1",        "supervised"),
    ("CONTROL lambda=0, pool r=10",            "runs_control_lambda0", "control_lambda0_r10"),
    ("MT r=10 (lambda=0.05)",                  "runs_final_v1",        "mean_teacher_r10"),
    ("CONTROL lambda=0, pool all-lateral",     "runs_control_lambda0", "control_lambda0_all_lateral"),
    ("MT all-lateral (lambda=0.05)",           "runs_final_v1",        "mean_teacher_all_lateral"),
]

print("{:<42}{:>3}{:>11}{:>9}".format("condicion", "n", "F1 medio", "sd"))
print("-" * 66)
ref = {}
for etiqueta, root, cond in filas:
    v = f1_of(root, cond)
    if not v:
        print("{:<42}{:>3}{:>20}".format(etiqueta, "-", "(sin corridas)"))
        continue
    m = st.mean(v)
    s = st.stdev(v) if len(v) > 1 else 0.0
    ref[cond] = m
    print(f"{etiqueta:<42}{len(v):>3}{m:>11.4f}{s:>9.4f}")

print()
sup = ref.get("supervised")
for pool, ctrl, real in [("r=10",        "control_lambda0_r10",        "mean_teacher_r10"),
                         ("all-lateral", "control_lambda0_all_lateral", "mean_teacher_all_lateral")]:
    if sup and ctrl in ref and real in ref:
        total = ref[real] - sup
        por_bn = ref[ctrl] - sup
        por_ssl = ref[real] - ref[ctrl]
        frac = (por_ssl / total * 100) if abs(total) > 1e-9 else float("nan")
        print(f"Pool {pool}:")
        print(f"  ganancia total sobre supervisado : {total:+.4f}")
        print(f"  atribuible al forward no etiq.   : {por_bn:+.4f}")
        print(f"  atribuible a la perdida SSL      : {por_ssl:+.4f}  ({frac:.0f}% del total)")
        print()


---
## Control en INCA al 10 % de etiquetas

`runs_inca_final_v1/mean_teacher_inca_r10_patient10` es el resultado mas solido del paper:
MT mejora al supervisado en los 23 pacientes de test, IC [+0.016, +0.030].

En UNM el control mostro que la perdida SSL no aporta nada detectable (la ganancia entera
viene del forward no etiquetado, que adapta las estadisticas de BatchNorm). Falta saber si
en INCA con pocas etiquetas ocurre lo mismo.

Referencias con las que comparar (media de 3 semillas, F1 de test):

| condicion | F1 |
|---|---|
| `supervised_inca_patient10` | 0.8437 |
| `mean_teacher_inca_r10_patient10` | 0.8654 |

Si el control queda cerca de 0.844, la perdida SSL **si** aporta con pocas etiquetas.
Si queda cerca de 0.865, el mecanismo es el mismo que en UNM.

**Ojo:** `semi_start_epoch` es 7 en INCA, no 15. Se mantiene igual que la corrida original.


In [ ]:
# === CONTROL INCA 1/3: control_lambda0_inca_r10_patient10/seed_0 ===
# Identico a runs_inca_final_v1/mean_teacher_inca_r10_patient10/seed_0
# SALVO lambda_u = 0.0 y exp_dir.
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()

# Paths (INCA)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["rotulos_dir"] = ""

_EXP_NAME = "control_lambda0_inca_r10_patient10"
_SEED     = 0
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_control_lambda0/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised — UNICA DIFERENCIA con la corrida final: lambda_u = 0.0
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True     # el forward no etiquetado y la EMA deben seguir ocurriendo
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.0      # <<<<<< el control
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 7        # INCA (en UNM es 15)
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40
cfg["eval_threshold"] = 0.5

# Pool no etiquetado y subconjunto de etiquetas al 10 %
cfg["unlabeled_subdir"]    = "unlabeled_r10"
cfg["labeled_subset_file"] = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset/label_fractions/patient_frac_10/stems.txt"

cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip    = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
_has_best    = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report  = len(glob.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all  = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado")

if not _skip_all:
    assert cfg["lambda_u"] == 0.0, "el control exige lambda_u = 0"
    assert cfg["use_semi"] is True, "use_semi debe seguir en True"
    assert cfg["semi_start_epoch"] == 7, "INCA usa semi_start = 7"
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )

    loaders = build_dataloaders(
        cfg, train_ds=train_ds, val_ds=val_ds, test_ds=test_ds,
        unlabeled_ds=unlabeled_ds, temporal_unlab_ds=temporal_unlab_ds,
    )

    if _eval_only:
        from src.models import create_model
        _m = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
        results = evaluate_checkpoint(cfg, _m, loaders, _best_path_skip, [])
    else:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg, artifacts["model"], loaders,
            artifacts["best_path"], artifacts["history"],
        )
    print(results)


In [ ]:
# === CONTROL INCA 2/3: control_lambda0_inca_r10_patient10/seed_1 ===
# Identico a runs_inca_final_v1/mean_teacher_inca_r10_patient10/seed_1
# SALVO lambda_u = 0.0 y exp_dir.
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()

# Paths (INCA)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["rotulos_dir"] = ""

_EXP_NAME = "control_lambda0_inca_r10_patient10"
_SEED     = 1
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_control_lambda0/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised — UNICA DIFERENCIA con la corrida final: lambda_u = 0.0
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True     # el forward no etiquetado y la EMA deben seguir ocurriendo
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.0      # <<<<<< el control
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 7        # INCA (en UNM es 15)
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40
cfg["eval_threshold"] = 0.5

# Pool no etiquetado y subconjunto de etiquetas al 10 %
cfg["unlabeled_subdir"]    = "unlabeled_r10"
cfg["labeled_subset_file"] = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset/label_fractions/patient_frac_10/stems.txt"

cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip    = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
_has_best    = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report  = len(glob.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all  = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado")

if not _skip_all:
    assert cfg["lambda_u"] == 0.0, "el control exige lambda_u = 0"
    assert cfg["use_semi"] is True, "use_semi debe seguir en True"
    assert cfg["semi_start_epoch"] == 7, "INCA usa semi_start = 7"
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )

    loaders = build_dataloaders(
        cfg, train_ds=train_ds, val_ds=val_ds, test_ds=test_ds,
        unlabeled_ds=unlabeled_ds, temporal_unlab_ds=temporal_unlab_ds,
    )

    if _eval_only:
        from src.models import create_model
        _m = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
        results = evaluate_checkpoint(cfg, _m, loaders, _best_path_skip, [])
    else:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg, artifacts["model"], loaders,
            artifacts["best_path"], artifacts["history"],
        )
    print(results)


In [ ]:
# === CONTROL INCA 3/3: control_lambda0_inca_r10_patient10/seed_2 ===
# Identico a runs_inca_final_v1/mean_teacher_inca_r10_patient10/seed_2
# SALVO lambda_u = 0.0 y exp_dir.
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()

# Paths (INCA)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["rotulos_dir"] = ""

_EXP_NAME = "control_lambda0_inca_r10_patient10"
_SEED     = 2
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_control_lambda0/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised — UNICA DIFERENCIA con la corrida final: lambda_u = 0.0
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True     # el forward no etiquetado y la EMA deben seguir ocurriendo
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.0      # <<<<<< el control
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 7        # INCA (en UNM es 15)
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40
cfg["eval_threshold"] = 0.5

# Pool no etiquetado y subconjunto de etiquetas al 10 %
cfg["unlabeled_subdir"]    = "unlabeled_r10"
cfg["labeled_subset_file"] = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset/label_fractions/patient_frac_10/stems.txt"

cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip    = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
_has_best    = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report  = len(glob.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all  = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado")

if not _skip_all:
    assert cfg["lambda_u"] == 0.0, "el control exige lambda_u = 0"
    assert cfg["use_semi"] is True, "use_semi debe seguir en True"
    assert cfg["semi_start_epoch"] == 7, "INCA usa semi_start = 7"
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )

    loaders = build_dataloaders(
        cfg, train_ds=train_ds, val_ds=val_ds, test_ds=test_ds,
        unlabeled_ds=unlabeled_ds, temporal_unlab_ds=temporal_unlab_ds,
    )

    if _eval_only:
        from src.models import create_model
        _m = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
        results = evaluate_checkpoint(cfg, _m, loaders, _best_path_skip, [])
    else:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg, artifacts["model"], loaders,
            artifacts["best_path"], artifacts["history"],
        )
    print(results)


### Resumen del control de INCA


In [ ]:
# === RESUMEN INCA (solo lectura) ===
import json, glob, os, statistics as st

ROOT = "/content/drive/MyDrive/UNM_vertebras_seg_v3"

def f1_of(root, cond):
    v = []
    for p in sorted(glob.glob(os.path.join(ROOT, root, cond, "seed_*", "*_run_report.json"))):
        x = json.load(open(p)).get("test_metrics", {}).get("sample_mean_f1")
        if x is not None:
            v.append(x)
    return v

filas = [
    ("supervisado 10% (sin datos no etiq.)", "runs_inca_final_v1",   "supervised_inca_patient10"),
    ("CONTROL lambda=0, r10, 10%",           "runs_control_lambda0", "control_lambda0_inca_r10_patient10"),
    ("MT r10 10% (lambda=0.05)",             "runs_inca_final_v1",   "mean_teacher_inca_r10_patient10"),
]

print("{:<40}{:>3}{:>11}{:>9}".format("condicion", "n", "F1 medio", "sd"))
print("-" * 63)
ref = {}
for etiqueta, root, cond in filas:
    v = f1_of(root, cond)
    if not v:
        print("{:<40}{:>3}{:>20}".format(etiqueta, "-", "(sin corridas)"))
        continue
    ref[cond] = st.mean(v)
    print("{:<40}{:>3}{:>11.4f}{:>9.4f}".format(
        etiqueta, len(v), st.mean(v), st.stdev(v) if len(v) > 1 else 0.0))

sup  = ref.get("supervised_inca_patient10")
ctrl = ref.get("control_lambda0_inca_r10_patient10")
mt   = ref.get("mean_teacher_inca_r10_patient10")
if sup and ctrl and mt:
    total, bn, ssl = mt - sup, ctrl - sup, mt - ctrl
    print()
    print(f"  ganancia total sobre supervisado : {total:+.4f}")
    print(f"  atribuible al forward no etiq.   : {bn:+.4f}")
    print(f"  atribuible a la perdida SSL      : {ssl:+.4f}", end="")
    print(f"  ({ssl/total*100:.0f}% del total)" if abs(total) > 1e-9 else "")
    print()
    print("  Comparar con UNM: la perdida SSL aporto +0.0052 (r10) y -0.0011 (all-lateral).")


---
## Bloque A — UNM r10, semillas 3 y 4 del control  (recomendado, 2 corridas)

`mean_teacher_r10` ya tiene 5 semillas. Con estas dos, la comparacion MT vs control queda
emparejada 5 contra 5 y no se toca ningun resultado del paper.

Con 3 semillas la diferencia MT - control era +0.0052 con sd 0.0157: demasiado ruido.


In [ ]:
# === UNM control lambda=0, r10, seed 3 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "control_lambda0_r10"
_SEED     = 3
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_control_lambda0/{_EXP_NAME}/seed_{_SEED}"

cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.0
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeling_r10_max0/images"

cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

_exp_dir = cfg["exp_dir"]
_best_path_skip    = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
_has_best    = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report  = len(glob.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all  = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado")

if not _skip_all:
    assert cfg["lambda_u"] == 0.0, "lambda_u no coincide con el diseno de esta celda"
    assert cfg["semi_start_epoch"] == 15, "semi_start no coincide con el dataset"
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )
    loaders = build_dataloaders(
        cfg, train_ds=train_ds, val_ds=val_ds, test_ds=test_ds,
        unlabeled_ds=unlabeled_ds, temporal_unlab_ds=temporal_unlab_ds,
    )

    if _eval_only:
        from src.models import create_model
        _m = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
        results = evaluate_checkpoint(cfg, _m, loaders, _best_path_skip, [])
    else:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg, artifacts["model"], loaders,
            artifacts["best_path"], artifacts["history"],
        )
    print(results)


In [ ]:
# === UNM control lambda=0, r10, seed 4 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "control_lambda0_r10"
_SEED     = 4
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_control_lambda0/{_EXP_NAME}/seed_{_SEED}"

cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.0
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeling_r10_max0/images"

cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

_exp_dir = cfg["exp_dir"]
_best_path_skip    = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
_has_best    = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report  = len(glob.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all  = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado")

if not _skip_all:
    assert cfg["lambda_u"] == 0.0, "lambda_u no coincide con el diseno de esta celda"
    assert cfg["semi_start_epoch"] == 15, "semi_start no coincide con el dataset"
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )
    loaders = build_dataloaders(
        cfg, train_ds=train_ds, val_ds=val_ds, test_ds=test_ds,
        unlabeled_ds=unlabeled_ds, temporal_unlab_ds=temporal_unlab_ds,
    )

    if _eval_only:
        from src.models import create_model
        _m = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
        results = evaluate_checkpoint(cfg, _m, loaders, _best_path_skip, [])
    else:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg, artifacts["model"], loaders,
            artifacts["best_path"], artifacts["history"],
        )
    print(results)


---
## Bloque B — INCA 10%, semillas 3 y 4  (opcional, 4 corridas)

En INCA el control y MT estan los dos en 3 semillas. Correr solo el control deja 5 contra 3
y **no se puede emparejar**, que es lo que le da fuerza al +0.0057.

Por eso este bloque corre las dos condiciones. Ojo: al pasar MT a 5 semillas cambian los
valores de INCA que hoy reporta la Tabla 4 del paper, que son de 3 semillas.


In [ ]:
# === INCA control lambda=0, 10%, seed 3 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["rotulos_dir"] = ""

_EXP_NAME = "control_lambda0_inca_r10_patient10"
_SEED     = 3
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_control_lambda0/{_EXP_NAME}/seed_{_SEED}"

cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.0
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 7
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeled_r10"
cfg["labeled_subset_file"] = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset/label_fractions/patient_frac_10/stems.txt"

cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

_exp_dir = cfg["exp_dir"]
_best_path_skip    = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
_has_best    = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report  = len(glob.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all  = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado")

if not _skip_all:
    assert cfg["lambda_u"] == 0.0, "lambda_u no coincide con el diseno de esta celda"
    assert cfg["semi_start_epoch"] == 7, "semi_start no coincide con el dataset"
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )
    loaders = build_dataloaders(
        cfg, train_ds=train_ds, val_ds=val_ds, test_ds=test_ds,
        unlabeled_ds=unlabeled_ds, temporal_unlab_ds=temporal_unlab_ds,
    )

    if _eval_only:
        from src.models import create_model
        _m = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
        results = evaluate_checkpoint(cfg, _m, loaders, _best_path_skip, [])
    else:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg, artifacts["model"], loaders,
            artifacts["best_path"], artifacts["history"],
        )
    print(results)


In [ ]:
# === INCA control lambda=0, 10%, seed 4 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["rotulos_dir"] = ""

_EXP_NAME = "control_lambda0_inca_r10_patient10"
_SEED     = 4
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_control_lambda0/{_EXP_NAME}/seed_{_SEED}"

cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.0
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 7
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeled_r10"
cfg["labeled_subset_file"] = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset/label_fractions/patient_frac_10/stems.txt"

cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

_exp_dir = cfg["exp_dir"]
_best_path_skip    = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
_has_best    = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report  = len(glob.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all  = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado")

if not _skip_all:
    assert cfg["lambda_u"] == 0.0, "lambda_u no coincide con el diseno de esta celda"
    assert cfg["semi_start_epoch"] == 7, "semi_start no coincide con el dataset"
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )
    loaders = build_dataloaders(
        cfg, train_ds=train_ds, val_ds=val_ds, test_ds=test_ds,
        unlabeled_ds=unlabeled_ds, temporal_unlab_ds=temporal_unlab_ds,
    )

    if _eval_only:
        from src.models import create_model
        _m = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
        results = evaluate_checkpoint(cfg, _m, loaders, _best_path_skip, [])
    else:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg, artifacts["model"], loaders,
            artifacts["best_path"], artifacts["history"],
        )
    print(results)


In [ ]:
# === INCA MT lambda=0.05, 10%, seed 3 (para poder emparejar) ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["rotulos_dir"] = ""

_EXP_NAME = "mean_teacher_inca_r10_patient10"
_SEED     = 3
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_inca_final_v1/{_EXP_NAME}/seed_{_SEED}"

cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 7
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeled_r10"
cfg["labeled_subset_file"] = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset/label_fractions/patient_frac_10/stems.txt"

cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

_exp_dir = cfg["exp_dir"]
_best_path_skip    = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
_has_best    = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report  = len(glob.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all  = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado")

if not _skip_all:
    assert cfg["lambda_u"] == 0.05, "lambda_u no coincide con el diseno de esta celda"
    assert cfg["semi_start_epoch"] == 7, "semi_start no coincide con el dataset"
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )
    loaders = build_dataloaders(
        cfg, train_ds=train_ds, val_ds=val_ds, test_ds=test_ds,
        unlabeled_ds=unlabeled_ds, temporal_unlab_ds=temporal_unlab_ds,
    )

    if _eval_only:
        from src.models import create_model
        _m = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
        results = evaluate_checkpoint(cfg, _m, loaders, _best_path_skip, [])
    else:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg, artifacts["model"], loaders,
            artifacts["best_path"], artifacts["history"],
        )
    print(results)


In [ ]:
# === INCA MT lambda=0.05, 10%, seed 4 (para poder emparejar) ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["rotulos_dir"] = ""

_EXP_NAME = "mean_teacher_inca_r10_patient10"
_SEED     = 4
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_inca_final_v1/{_EXP_NAME}/seed_{_SEED}"

cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 7
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeled_r10"
cfg["labeled_subset_file"] = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset/label_fractions/patient_frac_10/stems.txt"

cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

_exp_dir = cfg["exp_dir"]
_best_path_skip    = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
_has_best    = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report  = len(glob.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all  = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado")

if not _skip_all:
    assert cfg["lambda_u"] == 0.05, "lambda_u no coincide con el diseno de esta celda"
    assert cfg["semi_start_epoch"] == 7, "semi_start no coincide con el dataset"
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )
    loaders = build_dataloaders(
        cfg, train_ds=train_ds, val_ds=val_ds, test_ds=test_ds,
        unlabeled_ds=unlabeled_ds, temporal_unlab_ds=temporal_unlab_ds,
    )

    if _eval_only:
        from src.models import create_model
        _m = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
        results = evaluate_checkpoint(cfg, _m, loaders, _best_path_skip, [])
    else:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg, artifacts["model"], loaders,
            artifacts["best_path"], artifacts["history"],
        )
    print(results)


---
## Vistas alineadas: MT r10 sin HorizontalFlip

### Que se prueba

`get_weak_augmentation` y `get_strong_augmentation` aplican `HorizontalFlip` con p=0.1 y
p=0.15 **muestreados de forma independiente**. La probabilidad de que exactamente una de las
dos vistas quede espejada es 0.1x0.85 + 0.9x0.15 = **0.22**.

En ese 22 % la perdida compara la prediccion del profesor sobre una imagen con la del alumno
sobre su espejo, pixel a pixel. Le esta ensenando al alumno a poner la vertebra del lado
equivocado.

El control lambda=0 mostro que en UNM la perdida SSL aporta **+0.0052** sobre el forward, o
sea casi nada. La pregunta es si aporta poco porque SSL no sirve aca, o porque una de cada
cinco muestras que la alimentan esta mal.

### Como leerlo

Comparar contra el control lambda=0 de r10 (mismas semillas):

| condicion | F1 (semillas 0-2) |
|---|---|
| control lambda=0, r10 | 0.8442 |
| MT r10 con flip (paper) | 0.8494 |
| MT r10 SIN flip | ? |

- Si queda cerca de 0.849 -> el flip no era el problema; la perdida SSL realmente aporta poco.
- Si sube claramente -> el flip estaba degradando la senal, y hay que reportarlo.

### Salvedad

Quitar el flip cambia dos cosas a la vez: desaparece el desalineamiento **y** hay algo menos
de aumentacion. El efecto de lo segundo deberia ser menor, porque las perturbaciones
fotometricas de la vista fuerte se mantienen. Aislar solo el alineamiento exigiria compartir
el flip entre las dos vistas, y eso si obligaria a tocar el codigo.

**No se modifica `augmentations.py`.** La celda arma los pipelines y los pasa como argumento.


In [ ]:
# === VISTAS ALINEADAS 1/3: mean_teacher_r10_noflip/seed_0 ===
# Identico a runs_final_v1/mean_teacher_r10/seed_0 SALVO que las vistas debil y
# fuerte se construyen SIN HorizontalFlip. Todo lo demas igual, incluido lambda_u = 0.05.
import gc; gc.collect()
torch.cuda.empty_cache()

import albumentations as A
import cv2

cfg = get_default_config()
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "mean_teacher_r10_noflip"
_SEED     = 0
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_control_lambda0/{_EXP_NAME}/seed_{_SEED}"

cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.05     # el peso NORMAL, no el control
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeling_r10_max0/images"

cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip    = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
_has_best    = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report  = len(glob.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all  = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado")

if not _skip_all:
    print(summarize_config(cfg))

    # Aumentacion supervisada: sin cambios, la del repositorio
    train_tf = get_supervised_train_augmentation(cfg)

    # Vistas debil y fuerte: mismos parametros que augmentations.py, SIN HorizontalFlip.
    weak_tf = A.Compose([
        A.ShiftScaleRotate(
            shift_limit=0.01, scale_limit=0.02, rotate_limit=3,
            border_mode=cv2.BORDER_CONSTANT, value=0, p=0.5,
        ),
        # A.HorizontalFlip(p=0.1)  <- omitido a proposito
    ])
    strong_tf = A.Compose([
        A.ShiftScaleRotate(
            shift_limit=0.015, scale_limit=0.04, rotate_limit=6,
            border_mode=cv2.BORDER_CONSTANT, value=0, p=0.6,
        ),
        # A.HorizontalFlip(p=0.15)  <- omitido a proposito
        A.RandomBrightnessContrast(brightness_limit=0.12, contrast_limit=0.12, p=0.6),
        A.RandomGamma(gamma_limit=(88, 112), p=0.3),
        A.GaussNoise(var_limit=(4.0, 16.0), p=0.25),
    ])

    # comprobacion explicita: ningun flip en ninguna de las dos ramas
    _nombres = [type(t).__name__ for t in list(weak_tf.transforms) + list(strong_tf.transforms)]
    assert not any("Flip" in n for n in _nombres), f"quedo un flip: {_nombres}"
    assert cfg["lambda_u"] == 0.05, "este experimento usa el lambda normal, no el control"
    print("pipelines sin flip:", _nombres)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )
    loaders = build_dataloaders(
        cfg, train_ds=train_ds, val_ds=val_ds, test_ds=test_ds,
        unlabeled_ds=unlabeled_ds, temporal_unlab_ds=temporal_unlab_ds,
    )

    if _eval_only:
        from src.models import create_model
        _m = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
        results = evaluate_checkpoint(cfg, _m, loaders, _best_path_skip, [])
    else:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg, artifacts["model"], loaders,
            artifacts["best_path"], artifacts["history"],
        )
    print(results)


In [ ]:
# === VISTAS ALINEADAS 2/3: mean_teacher_r10_noflip/seed_1 ===
# Identico a runs_final_v1/mean_teacher_r10/seed_1 SALVO que las vistas debil y
# fuerte se construyen SIN HorizontalFlip. Todo lo demas igual, incluido lambda_u = 0.05.
import gc; gc.collect()
torch.cuda.empty_cache()

import albumentations as A
import cv2

cfg = get_default_config()
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "mean_teacher_r10_noflip"
_SEED     = 1
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_control_lambda0/{_EXP_NAME}/seed_{_SEED}"

cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.05     # el peso NORMAL, no el control
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeling_r10_max0/images"

cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip    = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
_has_best    = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report  = len(glob.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all  = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado")

if not _skip_all:
    print(summarize_config(cfg))

    # Aumentacion supervisada: sin cambios, la del repositorio
    train_tf = get_supervised_train_augmentation(cfg)

    # Vistas debil y fuerte: mismos parametros que augmentations.py, SIN HorizontalFlip.
    weak_tf = A.Compose([
        A.ShiftScaleRotate(
            shift_limit=0.01, scale_limit=0.02, rotate_limit=3,
            border_mode=cv2.BORDER_CONSTANT, value=0, p=0.5,
        ),
        # A.HorizontalFlip(p=0.1)  <- omitido a proposito
    ])
    strong_tf = A.Compose([
        A.ShiftScaleRotate(
            shift_limit=0.015, scale_limit=0.04, rotate_limit=6,
            border_mode=cv2.BORDER_CONSTANT, value=0, p=0.6,
        ),
        # A.HorizontalFlip(p=0.15)  <- omitido a proposito
        A.RandomBrightnessContrast(brightness_limit=0.12, contrast_limit=0.12, p=0.6),
        A.RandomGamma(gamma_limit=(88, 112), p=0.3),
        A.GaussNoise(var_limit=(4.0, 16.0), p=0.25),
    ])

    # comprobacion explicita: ningun flip en ninguna de las dos ramas
    _nombres = [type(t).__name__ for t in list(weak_tf.transforms) + list(strong_tf.transforms)]
    assert not any("Flip" in n for n in _nombres), f"quedo un flip: {_nombres}"
    assert cfg["lambda_u"] == 0.05, "este experimento usa el lambda normal, no el control"
    print("pipelines sin flip:", _nombres)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )
    loaders = build_dataloaders(
        cfg, train_ds=train_ds, val_ds=val_ds, test_ds=test_ds,
        unlabeled_ds=unlabeled_ds, temporal_unlab_ds=temporal_unlab_ds,
    )

    if _eval_only:
        from src.models import create_model
        _m = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
        results = evaluate_checkpoint(cfg, _m, loaders, _best_path_skip, [])
    else:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg, artifacts["model"], loaders,
            artifacts["best_path"], artifacts["history"],
        )
    print(results)


In [ ]:
# === VISTAS ALINEADAS 3/3: mean_teacher_r10_noflip/seed_2 ===
# Identico a runs_final_v1/mean_teacher_r10/seed_2 SALVO que las vistas debil y
# fuerte se construyen SIN HorizontalFlip. Todo lo demas igual, incluido lambda_u = 0.05.
import gc; gc.collect()
torch.cuda.empty_cache()

import albumentations as A
import cv2

cfg = get_default_config()
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "mean_teacher_r10_noflip"
_SEED     = 2
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_control_lambda0/{_EXP_NAME}/seed_{_SEED}"

cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.05     # el peso NORMAL, no el control
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeling_r10_max0/images"

cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip    = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
_has_best    = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report  = len(glob.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all  = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado")

if not _skip_all:
    print(summarize_config(cfg))

    # Aumentacion supervisada: sin cambios, la del repositorio
    train_tf = get_supervised_train_augmentation(cfg)

    # Vistas debil y fuerte: mismos parametros que augmentations.py, SIN HorizontalFlip.
    weak_tf = A.Compose([
        A.ShiftScaleRotate(
            shift_limit=0.01, scale_limit=0.02, rotate_limit=3,
            border_mode=cv2.BORDER_CONSTANT, value=0, p=0.5,
        ),
        # A.HorizontalFlip(p=0.1)  <- omitido a proposito
    ])
    strong_tf = A.Compose([
        A.ShiftScaleRotate(
            shift_limit=0.015, scale_limit=0.04, rotate_limit=6,
            border_mode=cv2.BORDER_CONSTANT, value=0, p=0.6,
        ),
        # A.HorizontalFlip(p=0.15)  <- omitido a proposito
        A.RandomBrightnessContrast(brightness_limit=0.12, contrast_limit=0.12, p=0.6),
        A.RandomGamma(gamma_limit=(88, 112), p=0.3),
        A.GaussNoise(var_limit=(4.0, 16.0), p=0.25),
    ])

    # comprobacion explicita: ningun flip en ninguna de las dos ramas
    _nombres = [type(t).__name__ for t in list(weak_tf.transforms) + list(strong_tf.transforms)]
    assert not any("Flip" in n for n in _nombres), f"quedo un flip: {_nombres}"
    assert cfg["lambda_u"] == 0.05, "este experimento usa el lambda normal, no el control"
    print("pipelines sin flip:", _nombres)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )
    loaders = build_dataloaders(
        cfg, train_ds=train_ds, val_ds=val_ds, test_ds=test_ds,
        unlabeled_ds=unlabeled_ds, temporal_unlab_ds=temporal_unlab_ds,
    )

    if _eval_only:
        from src.models import create_model
        _m = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
        results = evaluate_checkpoint(cfg, _m, loaders, _best_path_skip, [])
    else:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg, artifacts["model"], loaders,
            artifacts["best_path"], artifacts["history"],
        )
    print(results)


### Resumen del experimento sin flip


In [ ]:
# === RESUMEN: aporta mas la perdida SSL con las vistas alineadas? ===
import json, glob, os, statistics as st

ROOT = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
S = (0, 1, 2)

def por_semilla(root, cond):
    d = {}
    for p in sorted(glob.glob(os.path.join(ROOT, root, cond, "seed_*", "*_run_report.json"))):
        s = int(os.path.dirname(p).replace("\\", "/").split("seed_")[1])
        d[s] = json.load(open(p))["test_metrics"]["sample_mean_f1"]
    return d

sup  = por_semilla("runs_final_v1",        "supervised")
ctrl = por_semilla("runs_control_lambda0", "control_lambda0_r10")
mt   = por_semilla("runs_final_v1",        "mean_teacher_r10")
nf   = por_semilla("runs_control_lambda0", "mean_teacher_r10_noflip")

filas = [("supervisado", sup), ("control lambda=0", ctrl),
         ("MT con flip (paper)", mt), ("MT SIN flip", nf)]
print("{:<24}{:>10}{:>10}{:>10}{:>10}".format("condicion", "s0", "s1", "s2", "media"))
print("-" * 64)
for et, d in filas:
    if not d:
        print("{:<24}{:>40}".format(et, "(sin corridas)")); continue
    v = [d[i] for i in S if i in d]
    fila = "".join("{:>10.4f}".format(d[i]) if i in d else "{:>10}".format("--") for i in S)
    print("{:<24}{}{:>10.4f}".format(et, fila, st.mean(v)))

if nf and ctrl:
    con = [mt[i] - ctrl[i] for i in S if i in mt and i in ctrl]
    sin = [nf[i] - ctrl[i] for i in S if i in nf and i in ctrl]
    print()
    print("aporte de la perdida SSL (condicion - control lambda=0):")
    print("  con flip : {:+.4f}   por semilla {}".format(
        st.mean(con), [round(x, 4) for x in con]))
    print("  sin flip : {:+.4f}   por semilla {}".format(
        st.mean(sin), [round(x, 4) for x in sin]))
    print()
    print("  Si 'sin flip' es claramente mayor, el desalineamiento estaba degradando la senal.")


## ¿Es BatchNorm lo que hace que el pool importe?

Con `lambda_u = 0` los datos no etiquetados no aportan gradiente, y sin embargo
pasar de r10 (3.937 frames) a all-lateral (74.774) sube el F1. La única vía por
la que eso puede ocurrir: el forward del estudiante sobre no etiquetados se
ejecuta igual, en modo `train()`, y actualiza las estadísticas de las 100 capas
de BatchNorm.

`cfg["freeze_bn_on_unlabeled"] = True` pone el `momentum` de esas capas a 0
durante ese forward. Los buffers no se mueven y la normalización sigue usando
las estadísticas del lote, así que la salida del forward es idéntica
(verificado: diferencia máxima 0.0). No es lo mismo que `eval()`, que sí
cambiaría lo que la capa calcula.

**Baseline a batir.** Pareado en las semillas 0, 1 y 2, la diferencia
all-lateral − r10 con BN actualizándose es **+0,0171** (sd 0,0151, p = 0,188).
Ojo: no es significativa, y está dominada por la semilla 0. Con 3 semillas la
potencia para detectar un efecto de ese tamaño es del 7 %, por eso aquí van 5.

**Cómo leerlo.** Si con BN congelado la diferencia cae hacia 0, el mecanismo es
la normalización y la contribución sobre el tamaño del pool se reescribe con
explicación. Si se mantiene en +0,02, la hipótesis está mal y hay que buscar
otra causa.

Los resultados van a `runs_control_lambda0_bnfrozen/`. Nada de lo ya corrido
se toca.


### Bloque A · completar el control existente a 5 semillas

Sin BatchNorm congelado. Sólo añade las semillas 3 y 4 que le faltan a `control_lambda0_all_lateral`, para poder comparar 5 contra 5 y en pareado.


In [ ]:
# === UNM control lambda=0, all_lateral, seed 3  (completa a 5 semillas) ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "control_lambda0_all_lateral"
_SEED     = 3
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_control_lambda0/{_EXP_NAME}/seed_{_SEED}"

cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.0
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0


cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeling_all_lateral/images"

cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

_exp_dir = cfg["exp_dir"]
_best_path_skip    = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
_has_best    = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report  = len(glob.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all  = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado")

if not _skip_all:
    assert cfg["lambda_u"] == 0.0, "lambda_u no coincide con el diseno de esta celda"
    assert cfg.get("freeze_bn_on_unlabeled", False) is False, "esta celda es el control SIN congelar"
    assert cfg["semi_start_epoch"] == 15, "semi_start no coincide con el dataset"
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )
    loaders = build_dataloaders(
        cfg, train_ds=train_ds, val_ds=val_ds, test_ds=test_ds,
        unlabeled_ds=unlabeled_ds, temporal_unlab_ds=temporal_unlab_ds,
    )

    if _eval_only:
        from src.models import create_model
        _m = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
        results = evaluate_checkpoint(cfg, _m, loaders, _best_path_skip, [])
    else:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg, artifacts["model"], loaders,
            artifacts["best_path"], artifacts["history"],
        )
    print(results)


In [ ]:
# === UNM control lambda=0, all_lateral, seed 4  (completa a 5 semillas) ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "control_lambda0_all_lateral"
_SEED     = 4
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_control_lambda0/{_EXP_NAME}/seed_{_SEED}"

cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.0
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0


cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeling_all_lateral/images"

cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

_exp_dir = cfg["exp_dir"]
_best_path_skip    = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
_has_best    = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report  = len(glob.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all  = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado")

if not _skip_all:
    assert cfg["lambda_u"] == 0.0, "lambda_u no coincide con el diseno de esta celda"
    assert cfg.get("freeze_bn_on_unlabeled", False) is False, "esta celda es el control SIN congelar"
    assert cfg["semi_start_epoch"] == 15, "semi_start no coincide con el dataset"
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )
    loaders = build_dataloaders(
        cfg, train_ds=train_ds, val_ds=val_ds, test_ds=test_ds,
        unlabeled_ds=unlabeled_ds, temporal_unlab_ds=temporal_unlab_ds,
    )

    if _eval_only:
        from src.models import create_model
        _m = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
        results = evaluate_checkpoint(cfg, _m, loaders, _best_path_skip, [])
    else:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg, artifacts["model"], loaders,
            artifacts["best_path"], artifacts["history"],
        )
    print(results)


### Bloque B · λ = 0 con BatchNorm congelado

Es el experimento decisivo. 10 corridas.


In [ ]:
# === UNM lambda=0 + BN congelado, r10, seed 0 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "bnfrozen_lambda0_r10"
_SEED     = 0
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_control_lambda0_bnfrozen/{_EXP_NAME}/seed_{_SEED}"

cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.0
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# --- LA UNICA DIFERENCIA con control_lambda0_r10 ---
# Con esto, los forwards del estudiante sobre no etiquetados dejan de mover las
# estadisticas de BatchNorm. Separa lo que aporta la perdida de consistencia de
# lo que aporta que el pool actualice la normalizacion.
cfg["freeze_bn_on_unlabeled"] = True

cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeling_r10_max0/images"

cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

_exp_dir = cfg["exp_dir"]
_best_path_skip    = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
_has_best    = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report  = len(glob.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all  = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado")

if not _skip_all:
    assert cfg["lambda_u"] == 0.0, "lambda_u no coincide con el diseno de esta celda"
    assert cfg["freeze_bn_on_unlabeled"] is True, "esta celda exige BatchNorm congelado"
    assert cfg["semi_start_epoch"] == 15, "semi_start no coincide con el dataset"
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )
    loaders = build_dataloaders(
        cfg, train_ds=train_ds, val_ds=val_ds, test_ds=test_ds,
        unlabeled_ds=unlabeled_ds, temporal_unlab_ds=temporal_unlab_ds,
    )

    if _eval_only:
        from src.models import create_model
        _m = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
        results = evaluate_checkpoint(cfg, _m, loaders, _best_path_skip, [])
    else:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg, artifacts["model"], loaders,
            artifacts["best_path"], artifacts["history"],
        )
    print(results)


In [ ]:
# === UNM lambda=0 + BN congelado, r10, seed 1 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "bnfrozen_lambda0_r10"
_SEED     = 1
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_control_lambda0_bnfrozen/{_EXP_NAME}/seed_{_SEED}"

cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.0
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# --- LA UNICA DIFERENCIA con control_lambda0_r10 ---
# Con esto, los forwards del estudiante sobre no etiquetados dejan de mover las
# estadisticas de BatchNorm. Separa lo que aporta la perdida de consistencia de
# lo que aporta que el pool actualice la normalizacion.
cfg["freeze_bn_on_unlabeled"] = True

cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeling_r10_max0/images"

cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

_exp_dir = cfg["exp_dir"]
_best_path_skip    = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
_has_best    = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report  = len(glob.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all  = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado")

if not _skip_all:
    assert cfg["lambda_u"] == 0.0, "lambda_u no coincide con el diseno de esta celda"
    assert cfg["freeze_bn_on_unlabeled"] is True, "esta celda exige BatchNorm congelado"
    assert cfg["semi_start_epoch"] == 15, "semi_start no coincide con el dataset"
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )
    loaders = build_dataloaders(
        cfg, train_ds=train_ds, val_ds=val_ds, test_ds=test_ds,
        unlabeled_ds=unlabeled_ds, temporal_unlab_ds=temporal_unlab_ds,
    )

    if _eval_only:
        from src.models import create_model
        _m = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
        results = evaluate_checkpoint(cfg, _m, loaders, _best_path_skip, [])
    else:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg, artifacts["model"], loaders,
            artifacts["best_path"], artifacts["history"],
        )
    print(results)


In [ ]:
# === UNM lambda=0 + BN congelado, r10, seed 2 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "bnfrozen_lambda0_r10"
_SEED     = 2
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_control_lambda0_bnfrozen/{_EXP_NAME}/seed_{_SEED}"

cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.0
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# --- LA UNICA DIFERENCIA con control_lambda0_r10 ---
# Con esto, los forwards del estudiante sobre no etiquetados dejan de mover las
# estadisticas de BatchNorm. Separa lo que aporta la perdida de consistencia de
# lo que aporta que el pool actualice la normalizacion.
cfg["freeze_bn_on_unlabeled"] = True

cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeling_r10_max0/images"

cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

_exp_dir = cfg["exp_dir"]
_best_path_skip    = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
_has_best    = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report  = len(glob.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all  = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado")

if not _skip_all:
    assert cfg["lambda_u"] == 0.0, "lambda_u no coincide con el diseno de esta celda"
    assert cfg["freeze_bn_on_unlabeled"] is True, "esta celda exige BatchNorm congelado"
    assert cfg["semi_start_epoch"] == 15, "semi_start no coincide con el dataset"
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )
    loaders = build_dataloaders(
        cfg, train_ds=train_ds, val_ds=val_ds, test_ds=test_ds,
        unlabeled_ds=unlabeled_ds, temporal_unlab_ds=temporal_unlab_ds,
    )

    if _eval_only:
        from src.models import create_model
        _m = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
        results = evaluate_checkpoint(cfg, _m, loaders, _best_path_skip, [])
    else:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg, artifacts["model"], loaders,
            artifacts["best_path"], artifacts["history"],
        )
    print(results)


In [ ]:
# === UNM lambda=0 + BN congelado, r10, seed 3 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "bnfrozen_lambda0_r10"
_SEED     = 3
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_control_lambda0_bnfrozen/{_EXP_NAME}/seed_{_SEED}"

cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.0
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# --- LA UNICA DIFERENCIA con control_lambda0_r10 ---
# Con esto, los forwards del estudiante sobre no etiquetados dejan de mover las
# estadisticas de BatchNorm. Separa lo que aporta la perdida de consistencia de
# lo que aporta que el pool actualice la normalizacion.
cfg["freeze_bn_on_unlabeled"] = True

cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeling_r10_max0/images"

cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

_exp_dir = cfg["exp_dir"]
_best_path_skip    = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
_has_best    = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report  = len(glob.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all  = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado")

if not _skip_all:
    assert cfg["lambda_u"] == 0.0, "lambda_u no coincide con el diseno de esta celda"
    assert cfg["freeze_bn_on_unlabeled"] is True, "esta celda exige BatchNorm congelado"
    assert cfg["semi_start_epoch"] == 15, "semi_start no coincide con el dataset"
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )
    loaders = build_dataloaders(
        cfg, train_ds=train_ds, val_ds=val_ds, test_ds=test_ds,
        unlabeled_ds=unlabeled_ds, temporal_unlab_ds=temporal_unlab_ds,
    )

    if _eval_only:
        from src.models import create_model
        _m = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
        results = evaluate_checkpoint(cfg, _m, loaders, _best_path_skip, [])
    else:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg, artifacts["model"], loaders,
            artifacts["best_path"], artifacts["history"],
        )
    print(results)


In [ ]:
# === UNM lambda=0 + BN congelado, r10, seed 4 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "bnfrozen_lambda0_r10"
_SEED     = 4
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_control_lambda0_bnfrozen/{_EXP_NAME}/seed_{_SEED}"

cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.0
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# --- LA UNICA DIFERENCIA con control_lambda0_r10 ---
# Con esto, los forwards del estudiante sobre no etiquetados dejan de mover las
# estadisticas de BatchNorm. Separa lo que aporta la perdida de consistencia de
# lo que aporta que el pool actualice la normalizacion.
cfg["freeze_bn_on_unlabeled"] = True

cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeling_r10_max0/images"

cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

_exp_dir = cfg["exp_dir"]
_best_path_skip    = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
_has_best    = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report  = len(glob.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all  = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado")

if not _skip_all:
    assert cfg["lambda_u"] == 0.0, "lambda_u no coincide con el diseno de esta celda"
    assert cfg["freeze_bn_on_unlabeled"] is True, "esta celda exige BatchNorm congelado"
    assert cfg["semi_start_epoch"] == 15, "semi_start no coincide con el dataset"
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )
    loaders = build_dataloaders(
        cfg, train_ds=train_ds, val_ds=val_ds, test_ds=test_ds,
        unlabeled_ds=unlabeled_ds, temporal_unlab_ds=temporal_unlab_ds,
    )

    if _eval_only:
        from src.models import create_model
        _m = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
        results = evaluate_checkpoint(cfg, _m, loaders, _best_path_skip, [])
    else:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg, artifacts["model"], loaders,
            artifacts["best_path"], artifacts["history"],
        )
    print(results)


In [ ]:
# === UNM lambda=0 + BN congelado, all_lateral, seed 0 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "bnfrozen_lambda0_all_lateral"
_SEED     = 0
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_control_lambda0_bnfrozen/{_EXP_NAME}/seed_{_SEED}"

cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.0
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# --- LA UNICA DIFERENCIA con control_lambda0_all_lateral ---
# Con esto, los forwards del estudiante sobre no etiquetados dejan de mover las
# estadisticas de BatchNorm. Separa lo que aporta la perdida de consistencia de
# lo que aporta que el pool actualice la normalizacion.
cfg["freeze_bn_on_unlabeled"] = True

cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeling_all_lateral/images"

cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

_exp_dir = cfg["exp_dir"]
_best_path_skip    = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
_has_best    = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report  = len(glob.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all  = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado")

if not _skip_all:
    assert cfg["lambda_u"] == 0.0, "lambda_u no coincide con el diseno de esta celda"
    assert cfg["freeze_bn_on_unlabeled"] is True, "esta celda exige BatchNorm congelado"
    assert cfg["semi_start_epoch"] == 15, "semi_start no coincide con el dataset"
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )
    loaders = build_dataloaders(
        cfg, train_ds=train_ds, val_ds=val_ds, test_ds=test_ds,
        unlabeled_ds=unlabeled_ds, temporal_unlab_ds=temporal_unlab_ds,
    )

    if _eval_only:
        from src.models import create_model
        _m = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
        results = evaluate_checkpoint(cfg, _m, loaders, _best_path_skip, [])
    else:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg, artifacts["model"], loaders,
            artifacts["best_path"], artifacts["history"],
        )
    print(results)


In [ ]:
# === UNM lambda=0 + BN congelado, all_lateral, seed 1 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "bnfrozen_lambda0_all_lateral"
_SEED     = 1
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_control_lambda0_bnfrozen/{_EXP_NAME}/seed_{_SEED}"

cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.0
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# --- LA UNICA DIFERENCIA con control_lambda0_all_lateral ---
# Con esto, los forwards del estudiante sobre no etiquetados dejan de mover las
# estadisticas de BatchNorm. Separa lo que aporta la perdida de consistencia de
# lo que aporta que el pool actualice la normalizacion.
cfg["freeze_bn_on_unlabeled"] = True

cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeling_all_lateral/images"

cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

_exp_dir = cfg["exp_dir"]
_best_path_skip    = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
_has_best    = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report  = len(glob.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all  = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado")

if not _skip_all:
    assert cfg["lambda_u"] == 0.0, "lambda_u no coincide con el diseno de esta celda"
    assert cfg["freeze_bn_on_unlabeled"] is True, "esta celda exige BatchNorm congelado"
    assert cfg["semi_start_epoch"] == 15, "semi_start no coincide con el dataset"
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )
    loaders = build_dataloaders(
        cfg, train_ds=train_ds, val_ds=val_ds, test_ds=test_ds,
        unlabeled_ds=unlabeled_ds, temporal_unlab_ds=temporal_unlab_ds,
    )

    if _eval_only:
        from src.models import create_model
        _m = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
        results = evaluate_checkpoint(cfg, _m, loaders, _best_path_skip, [])
    else:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg, artifacts["model"], loaders,
            artifacts["best_path"], artifacts["history"],
        )
    print(results)


In [ ]:
# === UNM lambda=0 + BN congelado, all_lateral, seed 2 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "bnfrozen_lambda0_all_lateral"
_SEED     = 2
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_control_lambda0_bnfrozen/{_EXP_NAME}/seed_{_SEED}"

cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.0
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# --- LA UNICA DIFERENCIA con control_lambda0_all_lateral ---
# Con esto, los forwards del estudiante sobre no etiquetados dejan de mover las
# estadisticas de BatchNorm. Separa lo que aporta la perdida de consistencia de
# lo que aporta que el pool actualice la normalizacion.
cfg["freeze_bn_on_unlabeled"] = True

cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeling_all_lateral/images"

cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

_exp_dir = cfg["exp_dir"]
_best_path_skip    = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
_has_best    = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report  = len(glob.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all  = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado")

if not _skip_all:
    assert cfg["lambda_u"] == 0.0, "lambda_u no coincide con el diseno de esta celda"
    assert cfg["freeze_bn_on_unlabeled"] is True, "esta celda exige BatchNorm congelado"
    assert cfg["semi_start_epoch"] == 15, "semi_start no coincide con el dataset"
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )
    loaders = build_dataloaders(
        cfg, train_ds=train_ds, val_ds=val_ds, test_ds=test_ds,
        unlabeled_ds=unlabeled_ds, temporal_unlab_ds=temporal_unlab_ds,
    )

    if _eval_only:
        from src.models import create_model
        _m = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
        results = evaluate_checkpoint(cfg, _m, loaders, _best_path_skip, [])
    else:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg, artifacts["model"], loaders,
            artifacts["best_path"], artifacts["history"],
        )
    print(results)


In [ ]:
# === UNM lambda=0 + BN congelado, all_lateral, seed 3 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "bnfrozen_lambda0_all_lateral"
_SEED     = 3
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_control_lambda0_bnfrozen/{_EXP_NAME}/seed_{_SEED}"

cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.0
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# --- LA UNICA DIFERENCIA con control_lambda0_all_lateral ---
# Con esto, los forwards del estudiante sobre no etiquetados dejan de mover las
# estadisticas de BatchNorm. Separa lo que aporta la perdida de consistencia de
# lo que aporta que el pool actualice la normalizacion.
cfg["freeze_bn_on_unlabeled"] = True

cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeling_all_lateral/images"

cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

_exp_dir = cfg["exp_dir"]
_best_path_skip    = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
_has_best    = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report  = len(glob.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all  = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado")

if not _skip_all:
    assert cfg["lambda_u"] == 0.0, "lambda_u no coincide con el diseno de esta celda"
    assert cfg["freeze_bn_on_unlabeled"] is True, "esta celda exige BatchNorm congelado"
    assert cfg["semi_start_epoch"] == 15, "semi_start no coincide con el dataset"
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )
    loaders = build_dataloaders(
        cfg, train_ds=train_ds, val_ds=val_ds, test_ds=test_ds,
        unlabeled_ds=unlabeled_ds, temporal_unlab_ds=temporal_unlab_ds,
    )

    if _eval_only:
        from src.models import create_model
        _m = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
        results = evaluate_checkpoint(cfg, _m, loaders, _best_path_skip, [])
    else:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg, artifacts["model"], loaders,
            artifacts["best_path"], artifacts["history"],
        )
    print(results)


In [ ]:
# === UNM lambda=0 + BN congelado, all_lateral, seed 4 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "bnfrozen_lambda0_all_lateral"
_SEED     = 4
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_control_lambda0_bnfrozen/{_EXP_NAME}/seed_{_SEED}"

cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.0
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# --- LA UNICA DIFERENCIA con control_lambda0_all_lateral ---
# Con esto, los forwards del estudiante sobre no etiquetados dejan de mover las
# estadisticas de BatchNorm. Separa lo que aporta la perdida de consistencia de
# lo que aporta que el pool actualice la normalizacion.
cfg["freeze_bn_on_unlabeled"] = True

cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeling_all_lateral/images"

cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

_exp_dir = cfg["exp_dir"]
_best_path_skip    = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
_has_best    = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report  = len(glob.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all  = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado")

if not _skip_all:
    assert cfg["lambda_u"] == 0.0, "lambda_u no coincide con el diseno de esta celda"
    assert cfg["freeze_bn_on_unlabeled"] is True, "esta celda exige BatchNorm congelado"
    assert cfg["semi_start_epoch"] == 15, "semi_start no coincide con el dataset"
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )
    loaders = build_dataloaders(
        cfg, train_ds=train_ds, val_ds=val_ds, test_ds=test_ds,
        unlabeled_ds=unlabeled_ds, temporal_unlab_ds=temporal_unlab_ds,
    )

    if _eval_only:
        from src.models import create_model
        _m = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
        results = evaluate_checkpoint(cfg, _m, loaders, _best_path_skip, [])
    else:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg, artifacts["model"], loaders,
            artifacts["best_path"], artifacts["history"],
        )
    print(results)


### Bloque C · λ = 0,05 con BatchNorm congelado

Completa el cuadro 2×2 y responde la pregunta inmediata de un revisor: ¿qué hace la pérdida SSL cuando la normalización no contamina? 10 corridas.


In [ ]:
# === UNM lambda=0.05 + BN congelado, r10, seed 0 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "bnfrozen_lambda005_r10"
_SEED     = 0
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_control_lambda0_bnfrozen/{_EXP_NAME}/seed_{_SEED}"

cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# --- LA UNICA DIFERENCIA con mean_teacher_r10 ---
# Con esto, los forwards del estudiante sobre no etiquetados dejan de mover las
# estadisticas de BatchNorm. Separa lo que aporta la perdida de consistencia de
# lo que aporta que el pool actualice la normalizacion.
cfg["freeze_bn_on_unlabeled"] = True

cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeling_r10_max0/images"

cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

_exp_dir = cfg["exp_dir"]
_best_path_skip    = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
_has_best    = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report  = len(glob.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all  = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado")

if not _skip_all:
    assert cfg["lambda_u"] == 0.05, "lambda_u no coincide con el diseno de esta celda"
    assert cfg["freeze_bn_on_unlabeled"] is True, "esta celda exige BatchNorm congelado"
    assert cfg["semi_start_epoch"] == 15, "semi_start no coincide con el dataset"
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )
    loaders = build_dataloaders(
        cfg, train_ds=train_ds, val_ds=val_ds, test_ds=test_ds,
        unlabeled_ds=unlabeled_ds, temporal_unlab_ds=temporal_unlab_ds,
    )

    if _eval_only:
        from src.models import create_model
        _m = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
        results = evaluate_checkpoint(cfg, _m, loaders, _best_path_skip, [])
    else:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg, artifacts["model"], loaders,
            artifacts["best_path"], artifacts["history"],
        )
    print(results)


In [ ]:
# === UNM lambda=0.05 + BN congelado, r10, seed 1 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "bnfrozen_lambda005_r10"
_SEED     = 1
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_control_lambda0_bnfrozen/{_EXP_NAME}/seed_{_SEED}"

cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# --- LA UNICA DIFERENCIA con mean_teacher_r10 ---
# Con esto, los forwards del estudiante sobre no etiquetados dejan de mover las
# estadisticas de BatchNorm. Separa lo que aporta la perdida de consistencia de
# lo que aporta que el pool actualice la normalizacion.
cfg["freeze_bn_on_unlabeled"] = True

cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeling_r10_max0/images"

cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

_exp_dir = cfg["exp_dir"]
_best_path_skip    = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
_has_best    = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report  = len(glob.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all  = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado")

if not _skip_all:
    assert cfg["lambda_u"] == 0.05, "lambda_u no coincide con el diseno de esta celda"
    assert cfg["freeze_bn_on_unlabeled"] is True, "esta celda exige BatchNorm congelado"
    assert cfg["semi_start_epoch"] == 15, "semi_start no coincide con el dataset"
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )
    loaders = build_dataloaders(
        cfg, train_ds=train_ds, val_ds=val_ds, test_ds=test_ds,
        unlabeled_ds=unlabeled_ds, temporal_unlab_ds=temporal_unlab_ds,
    )

    if _eval_only:
        from src.models import create_model
        _m = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
        results = evaluate_checkpoint(cfg, _m, loaders, _best_path_skip, [])
    else:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg, artifacts["model"], loaders,
            artifacts["best_path"], artifacts["history"],
        )
    print(results)


In [ ]:
# === UNM lambda=0.05 + BN congelado, r10, seed 2 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "bnfrozen_lambda005_r10"
_SEED     = 2
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_control_lambda0_bnfrozen/{_EXP_NAME}/seed_{_SEED}"

cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# --- LA UNICA DIFERENCIA con mean_teacher_r10 ---
# Con esto, los forwards del estudiante sobre no etiquetados dejan de mover las
# estadisticas de BatchNorm. Separa lo que aporta la perdida de consistencia de
# lo que aporta que el pool actualice la normalizacion.
cfg["freeze_bn_on_unlabeled"] = True

cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeling_r10_max0/images"

cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

_exp_dir = cfg["exp_dir"]
_best_path_skip    = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
_has_best    = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report  = len(glob.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all  = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado")

if not _skip_all:
    assert cfg["lambda_u"] == 0.05, "lambda_u no coincide con el diseno de esta celda"
    assert cfg["freeze_bn_on_unlabeled"] is True, "esta celda exige BatchNorm congelado"
    assert cfg["semi_start_epoch"] == 15, "semi_start no coincide con el dataset"
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )
    loaders = build_dataloaders(
        cfg, train_ds=train_ds, val_ds=val_ds, test_ds=test_ds,
        unlabeled_ds=unlabeled_ds, temporal_unlab_ds=temporal_unlab_ds,
    )

    if _eval_only:
        from src.models import create_model
        _m = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
        results = evaluate_checkpoint(cfg, _m, loaders, _best_path_skip, [])
    else:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg, artifacts["model"], loaders,
            artifacts["best_path"], artifacts["history"],
        )
    print(results)


In [ ]:
# === UNM lambda=0.05 + BN congelado, r10, seed 3 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "bnfrozen_lambda005_r10"
_SEED     = 3
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_control_lambda0_bnfrozen/{_EXP_NAME}/seed_{_SEED}"

cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# --- LA UNICA DIFERENCIA con mean_teacher_r10 ---
# Con esto, los forwards del estudiante sobre no etiquetados dejan de mover las
# estadisticas de BatchNorm. Separa lo que aporta la perdida de consistencia de
# lo que aporta que el pool actualice la normalizacion.
cfg["freeze_bn_on_unlabeled"] = True

cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeling_r10_max0/images"

cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

_exp_dir = cfg["exp_dir"]
_best_path_skip    = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
_has_best    = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report  = len(glob.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all  = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado")

if not _skip_all:
    assert cfg["lambda_u"] == 0.05, "lambda_u no coincide con el diseno de esta celda"
    assert cfg["freeze_bn_on_unlabeled"] is True, "esta celda exige BatchNorm congelado"
    assert cfg["semi_start_epoch"] == 15, "semi_start no coincide con el dataset"
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )
    loaders = build_dataloaders(
        cfg, train_ds=train_ds, val_ds=val_ds, test_ds=test_ds,
        unlabeled_ds=unlabeled_ds, temporal_unlab_ds=temporal_unlab_ds,
    )

    if _eval_only:
        from src.models import create_model
        _m = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
        results = evaluate_checkpoint(cfg, _m, loaders, _best_path_skip, [])
    else:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg, artifacts["model"], loaders,
            artifacts["best_path"], artifacts["history"],
        )
    print(results)


In [ ]:
# === UNM lambda=0.05 + BN congelado, r10, seed 4 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "bnfrozen_lambda005_r10"
_SEED     = 4
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_control_lambda0_bnfrozen/{_EXP_NAME}/seed_{_SEED}"

cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# --- LA UNICA DIFERENCIA con mean_teacher_r10 ---
# Con esto, los forwards del estudiante sobre no etiquetados dejan de mover las
# estadisticas de BatchNorm. Separa lo que aporta la perdida de consistencia de
# lo que aporta que el pool actualice la normalizacion.
cfg["freeze_bn_on_unlabeled"] = True

cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeling_r10_max0/images"

cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

_exp_dir = cfg["exp_dir"]
_best_path_skip    = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
_has_best    = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report  = len(glob.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all  = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado")

if not _skip_all:
    assert cfg["lambda_u"] == 0.05, "lambda_u no coincide con el diseno de esta celda"
    assert cfg["freeze_bn_on_unlabeled"] is True, "esta celda exige BatchNorm congelado"
    assert cfg["semi_start_epoch"] == 15, "semi_start no coincide con el dataset"
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )
    loaders = build_dataloaders(
        cfg, train_ds=train_ds, val_ds=val_ds, test_ds=test_ds,
        unlabeled_ds=unlabeled_ds, temporal_unlab_ds=temporal_unlab_ds,
    )

    if _eval_only:
        from src.models import create_model
        _m = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
        results = evaluate_checkpoint(cfg, _m, loaders, _best_path_skip, [])
    else:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg, artifacts["model"], loaders,
            artifacts["best_path"], artifacts["history"],
        )
    print(results)


In [ ]:
# === UNM lambda=0.05 + BN congelado, all_lateral, seed 0 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "bnfrozen_lambda005_all_lateral"
_SEED     = 0
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_control_lambda0_bnfrozen/{_EXP_NAME}/seed_{_SEED}"

cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# --- LA UNICA DIFERENCIA con mean_teacher_all_lateral ---
# Con esto, los forwards del estudiante sobre no etiquetados dejan de mover las
# estadisticas de BatchNorm. Separa lo que aporta la perdida de consistencia de
# lo que aporta que el pool actualice la normalizacion.
cfg["freeze_bn_on_unlabeled"] = True

cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeling_all_lateral/images"

cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

_exp_dir = cfg["exp_dir"]
_best_path_skip    = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
_has_best    = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report  = len(glob.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all  = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado")

if not _skip_all:
    assert cfg["lambda_u"] == 0.05, "lambda_u no coincide con el diseno de esta celda"
    assert cfg["freeze_bn_on_unlabeled"] is True, "esta celda exige BatchNorm congelado"
    assert cfg["semi_start_epoch"] == 15, "semi_start no coincide con el dataset"
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )
    loaders = build_dataloaders(
        cfg, train_ds=train_ds, val_ds=val_ds, test_ds=test_ds,
        unlabeled_ds=unlabeled_ds, temporal_unlab_ds=temporal_unlab_ds,
    )

    if _eval_only:
        from src.models import create_model
        _m = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
        results = evaluate_checkpoint(cfg, _m, loaders, _best_path_skip, [])
    else:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg, artifacts["model"], loaders,
            artifacts["best_path"], artifacts["history"],
        )
    print(results)


In [ ]:
# === UNM lambda=0.05 + BN congelado, all_lateral, seed 1 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "bnfrozen_lambda005_all_lateral"
_SEED     = 1
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_control_lambda0_bnfrozen/{_EXP_NAME}/seed_{_SEED}"

cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# --- LA UNICA DIFERENCIA con mean_teacher_all_lateral ---
# Con esto, los forwards del estudiante sobre no etiquetados dejan de mover las
# estadisticas de BatchNorm. Separa lo que aporta la perdida de consistencia de
# lo que aporta que el pool actualice la normalizacion.
cfg["freeze_bn_on_unlabeled"] = True

cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeling_all_lateral/images"

cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

_exp_dir = cfg["exp_dir"]
_best_path_skip    = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
_has_best    = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report  = len(glob.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all  = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado")

if not _skip_all:
    assert cfg["lambda_u"] == 0.05, "lambda_u no coincide con el diseno de esta celda"
    assert cfg["freeze_bn_on_unlabeled"] is True, "esta celda exige BatchNorm congelado"
    assert cfg["semi_start_epoch"] == 15, "semi_start no coincide con el dataset"
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )
    loaders = build_dataloaders(
        cfg, train_ds=train_ds, val_ds=val_ds, test_ds=test_ds,
        unlabeled_ds=unlabeled_ds, temporal_unlab_ds=temporal_unlab_ds,
    )

    if _eval_only:
        from src.models import create_model
        _m = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
        results = evaluate_checkpoint(cfg, _m, loaders, _best_path_skip, [])
    else:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg, artifacts["model"], loaders,
            artifacts["best_path"], artifacts["history"],
        )
    print(results)


In [ ]:
# === UNM lambda=0.05 + BN congelado, all_lateral, seed 2 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "bnfrozen_lambda005_all_lateral"
_SEED     = 2
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_control_lambda0_bnfrozen/{_EXP_NAME}/seed_{_SEED}"

cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# --- LA UNICA DIFERENCIA con mean_teacher_all_lateral ---
# Con esto, los forwards del estudiante sobre no etiquetados dejan de mover las
# estadisticas de BatchNorm. Separa lo que aporta la perdida de consistencia de
# lo que aporta que el pool actualice la normalizacion.
cfg["freeze_bn_on_unlabeled"] = True

cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeling_all_lateral/images"

cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

_exp_dir = cfg["exp_dir"]
_best_path_skip    = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
_has_best    = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report  = len(glob.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all  = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado")

if not _skip_all:
    assert cfg["lambda_u"] == 0.05, "lambda_u no coincide con el diseno de esta celda"
    assert cfg["freeze_bn_on_unlabeled"] is True, "esta celda exige BatchNorm congelado"
    assert cfg["semi_start_epoch"] == 15, "semi_start no coincide con el dataset"
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )
    loaders = build_dataloaders(
        cfg, train_ds=train_ds, val_ds=val_ds, test_ds=test_ds,
        unlabeled_ds=unlabeled_ds, temporal_unlab_ds=temporal_unlab_ds,
    )

    if _eval_only:
        from src.models import create_model
        _m = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
        results = evaluate_checkpoint(cfg, _m, loaders, _best_path_skip, [])
    else:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg, artifacts["model"], loaders,
            artifacts["best_path"], artifacts["history"],
        )
    print(results)


In [ ]:
# === UNM lambda=0.05 + BN congelado, all_lateral, seed 3 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "bnfrozen_lambda005_all_lateral"
_SEED     = 3
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_control_lambda0_bnfrozen/{_EXP_NAME}/seed_{_SEED}"

cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# --- LA UNICA DIFERENCIA con mean_teacher_all_lateral ---
# Con esto, los forwards del estudiante sobre no etiquetados dejan de mover las
# estadisticas de BatchNorm. Separa lo que aporta la perdida de consistencia de
# lo que aporta que el pool actualice la normalizacion.
cfg["freeze_bn_on_unlabeled"] = True

cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeling_all_lateral/images"

cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

_exp_dir = cfg["exp_dir"]
_best_path_skip    = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
_has_best    = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report  = len(glob.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all  = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado")

if not _skip_all:
    assert cfg["lambda_u"] == 0.05, "lambda_u no coincide con el diseno de esta celda"
    assert cfg["freeze_bn_on_unlabeled"] is True, "esta celda exige BatchNorm congelado"
    assert cfg["semi_start_epoch"] == 15, "semi_start no coincide con el dataset"
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )
    loaders = build_dataloaders(
        cfg, train_ds=train_ds, val_ds=val_ds, test_ds=test_ds,
        unlabeled_ds=unlabeled_ds, temporal_unlab_ds=temporal_unlab_ds,
    )

    if _eval_only:
        from src.models import create_model
        _m = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
        results = evaluate_checkpoint(cfg, _m, loaders, _best_path_skip, [])
    else:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg, artifacts["model"], loaders,
            artifacts["best_path"], artifacts["history"],
        )
    print(results)


In [ ]:
# === UNM lambda=0.05 + BN congelado, all_lateral, seed 4 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "bnfrozen_lambda005_all_lateral"
_SEED     = 4
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_control_lambda0_bnfrozen/{_EXP_NAME}/seed_{_SEED}"

cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# --- LA UNICA DIFERENCIA con mean_teacher_all_lateral ---
# Con esto, los forwards del estudiante sobre no etiquetados dejan de mover las
# estadisticas de BatchNorm. Separa lo que aporta la perdida de consistencia de
# lo que aporta que el pool actualice la normalizacion.
cfg["freeze_bn_on_unlabeled"] = True

cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeling_all_lateral/images"

cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

_exp_dir = cfg["exp_dir"]
_best_path_skip    = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
_has_best    = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report  = len(glob.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all  = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado")

if not _skip_all:
    assert cfg["lambda_u"] == 0.05, "lambda_u no coincide con el diseno de esta celda"
    assert cfg["freeze_bn_on_unlabeled"] is True, "esta celda exige BatchNorm congelado"
    assert cfg["semi_start_epoch"] == 15, "semi_start no coincide con el dataset"
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )
    loaders = build_dataloaders(
        cfg, train_ds=train_ds, val_ds=val_ds, test_ds=test_ds,
        unlabeled_ds=unlabeled_ds, temporal_unlab_ds=temporal_unlab_ds,
    )

    if _eval_only:
        from src.models import create_model
        _m = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
        results = evaluate_checkpoint(cfg, _m, loaders, _best_path_skip, [])
    else:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg, artifacts["model"], loaders,
            artifacts["best_path"], artifacts["history"],
        )
    print(results)


In [ ]:
# === Resumen: se cae el efecto del pool al congelar BatchNorm? ===
import glob, json, os, statistics as st
from itertools import product

RAIZ = "/content/drive/MyDrive/UNM_vertebras_seg_v3"

def f1_por_semilla(carpeta, exp):
    out = {}
    for d in sorted(glob.glob(os.path.join(RAIZ, carpeta, exp, "seed_*"))):
        for p in glob.glob(os.path.join(d, "*_run_report.json")):
            with open(p, encoding="utf-8") as f:
                j = json.load(f)
            v = j.get("test_metrics", {}).get("sample_mean_f1")
            if v is not None:
                out[os.path.basename(d)] = float(v)
            break
    return out

def pareado(a, b):
    """Diferencia b - a solo en las semillas que tienen las dos."""
    comunes = sorted(set(a) & set(b))
    d = [b[s] - a[s] for s in comunes]
    if not d:
        return None
    media = st.mean(d)
    sd = st.stdev(d) if len(d) > 1 else 0.0
    return comunes, d, media, sd

CASOS = [
    ("BN se actualiza  (ya corrido)", "runs_control_lambda0",
     "control_lambda0_r10", "control_lambda0_all_lateral"),
    ("BN congelado     (nuevo)", "runs_control_lambda0_bnfrozen",
     "bnfrozen_lambda0_r10", "bnfrozen_lambda0_all_lateral"),
]

print("EFECTO DEL POOL (all_lateral - r10), con lambda_u = 0\n")
for etiqueta, carpeta, exp_r10, exp_all in CASOS:
    a, b = f1_por_semilla(carpeta, exp_r10), f1_por_semilla(carpeta, exp_all)
    print(f"{etiqueta}")
    print(f"   r10          n={len(a)}  " +
          "  ".join(f"{k[-1]}:{v:.4f}" for k, v in sorted(a.items())))
    print(f"   all_lateral  n={len(b)}  " +
          "  ".join(f"{k[-1]}:{v:.4f}" for k, v in sorted(b.items())))
    r = pareado(a, b)
    if r:
        comunes, d, media, sd = r
        print(f"   PAREADO en {len(comunes)} semillas: {media:+.4f}  (sd {sd:.4f})")
        print("   deltas: " + "  ".join(f"{x:+.4f}" for x in d))
        pares = list(product(b.values(), a.values()))
        gana = sum(1 for x, y in pares if x > y)
        print(f"   separacion: {gana}/{len(pares)} pares")
    else:
        print("   (aun sin semillas comunes)")
    print()

print("Lectura: si el pareado con BN congelado cae hacia 0, el mecanismo era")
print("BatchNorm. Si sigue cerca de +0.017, la hipotesis estaba mal.\n")

print("APORTE DE LA PERDIDA SSL, con BatchNorm congelado (cuadro 2x2)\n")
for pool in ("r10", "all_lateral"):
    a = f1_por_semilla("runs_control_lambda0_bnfrozen", f"bnfrozen_lambda0_{pool}")
    b = f1_por_semilla("runs_control_lambda0_bnfrozen", f"bnfrozen_lambda005_{pool}")
    r = pareado(a, b)
    if r:
        comunes, d, media, sd = r
        print(f"   {pool}: lambda 0.05 - lambda 0 = {media:+.4f} "
              f"(sd {sd:.4f}, n={len(comunes)})")
    else:
        print(f"   {pool}: aun sin datos")
